## 1. Resume_text_extraction :

In [50]:
import pdfplumber
import pandas as pd
from sentence_transformers import SentenceTransformer,util

In [51]:
def extract_resume_text(pdf_name):
    resume_text = ""

    with pdfplumber.open(pdf_name) as pdf:
     for page in pdf.pages:
        text = page.extract_text()

        if text:
            resume_text += text +'\n'
    return resume_text       

resume = extract_resume_text("Thanesh_Resume.pdf")

## 2. Cleaning the text :

In [52]:
import re

In [53]:
def clean_resume_text(resume_text):
    text = re.sub(r"\s+"," ",resume_text)                           # remove extra spaces
       
    text = re.sub(r"\S+@\S+"," ",text)                              # removes email as it will not be needed for us
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)              # removes unwanted links eg:my linked in profile
    text = re.sub(r"\+?\d[\d\s\-()]{8,}\d"," ", text)               # remove phone nos as they are not uch needed
    text = re.sub(r"[^a-zA-Z0-9.,()#+/\-]"," ",text)                #removes unwanted spl caharacters
    text = re.sub(r"\s+"," ",text)                                  # again removing extra spaces
    text = text.strip()                                             # removing trailing and ending spaces
    return text

resume_cleaned = clean_resume_text(resume)

## The above mentioned is our cleaned resume .

In [55]:
requirements = pd.read_csv("Requirement.csv")
requirements

,Domain,Description,Required_skills
0,Data Scientist,We are seeking a motivated Data Scientist to j...,"Python, SQL, Machine Learning, Statistics, Pan..."
1,Data Analyst,We are hiring a Data Analyst with strong analy...,"Python, SQL, Excel, Power BI, Statistics, Pand..."
2,Front-End Developer,We are looking for a Front-End Developer with ...,"HTML, CSS, JavaScript, React.js, Bootstrap, RE..."
3,Full Stack Developer,We are seeking a Full Stack Developer with kno...,"HTML, CSS, JavaScript, React.js, Node.js, Expr..."
4,Python Developer,We are hiring a Python Developer with strong p...,"Python, OOP, SQL, Flask, Django, REST API, Pan..."


### Particularly slicing the domain description .

In [95]:
Selected_Domain = "Data Scientist"
selected_row = requirements[requirements["Domain"] == Selected_Domain]

if selected_row.empty:
    print("Domain was not found.")
else :    
    job_description = selected_row["Description"].iloc[0]
    
    ## 1. Resume match Score :
    ## Validating using sentence transformer .

    model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

    resume_embedding =model.encode(
    resume_cleaned,                        ## Converting resume text to vectors
    convert_to_tensor=True
)

    job_embedding = model.encode(
    job_description,                       ## Converting description text to vectors
    convert_to_tensor=True
)

    similarity_score = util.cos_sim(                 ## finding similarity btw two values
    resume_embedding,
    job_embedding
).item()

    resume_score = round(similarity_score * 100 , 2)

    ## 2. Skill match score :

    
    job_skills = selected_row["Required_skills"].iloc[0]

    job_req_skills = [
        skill.strip()
        for skill in job_skills.split(",")           ## Converts each and every word into a stringthat means breaks it
    ]
    resume_lower = resume_cleaned.lower()

    Matched_skills = []
    UnMatched_skills = []

    for skill in job_req_skills:
        if skill.lower() in resume_lower:
            Matched_skills.append(skill)
        else:
            UnMatched_skills.append(skill)

    ## 3. Calculating skill Score :

    skill_score = round(len(Matched_skills) / len(job_req_skills) * 100,2)


    ## 4. Overall Score :

    Overall_score = round((0.8 * resume_score) + (0.2 * skill_score) , 2)
    

     

  
    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Changes made from above , converted all functions into single function .

In [101]:
def analyze_resume(resume_cleaned, Selected_Domain):

    selected_row = requirements[
        requirements["Domain"] == Selected_Domain
    ]

    if selected_row.empty:
        return None

    job_description = selected_row["Description"].iloc[0]

    # Your existing resume-score code
    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    resume_embedding = model.encode(
        resume_cleaned,
        convert_to_tensor=True
    )

    job_embedding = model.encode(
        job_description,
        convert_to_tensor=True
    )

    similarity_score = util.cos_sim(
        resume_embedding,
        job_embedding
    ).item()

    resume_score = round(
        similarity_score * 100,
        2
    )

    # Your existing skill-score code
    job_skills = selected_row[
        "Required_skills"
    ].iloc[0]

    job_req_skills = [
        skill.strip()
        for skill in job_skills.split(",")
    ]

    resume_lower = resume_cleaned.lower()

    Matched_skills = []
    UnMatched_skills = []

    for skill in job_req_skills:

        if skill.lower() in resume_lower:
            Matched_skills.append(skill)

        else:
            UnMatched_skills.append(skill)

    skill_score = round(
        len(Matched_skills)
        / len(job_req_skills)
        * 100,
        2
    )

    Overall_score = round(
        (0.8 * resume_score)
        + (0.2 * skill_score),
        2
    )

    return (
        resume_score,
        skill_score,
        Overall_score,
        Matched_skills,
        UnMatched_skills
    )